使用一个聪明的模型提取答案，按照提取LLM的格式输出
只运行单一的json
限制回答问题的LLM的输出，限定了给定格式

In [14]:
import json
import os
from PIL import Image
import matplotlib.pyplot as plt
import requests
import re
import ast
import string
import textwrap
import base64
import random
import time
from openai import OpenAI
import io
import matplotlib.patches as mpatches

In [15]:
def generate_prompt_q1(sample):
    question_gen = sample["question_gen"]
    items = sample["items"]
    caption_map = sample["caption"]
    
    # caption
    lines = []
    for item in items:
        caption_text = caption_map.get(item, "")
        lines.append(f"the caption of {item}: {caption_text}")
    caption = "\n".join(lines)
    
    prompt = f"""
You are a data science assistant specialized in understanding charts and tables.

You are given the following items: {items}.
Each item has a corresponding caption:
{caption}

Question:
{question_gen}

Instructions:
Analyze the provided items to answer the question.
"""

    return prompt

In [16]:
def generate_prompt_q2(sample):
    items = sample["items"]
    caption_map = sample["caption"]
    question_gen = sample["question_gen"]
    question_comp = sample["question_comp"]
    answer_type = sample["answer_type"]

    # 输出格式规则
    if answer_type == "numeric":
        format_rule = "a single number"
    elif answer_type == "numeric_list":
        format_rule = "[a, b, c]"
    elif answer_type == "choice":
        format_rule = "A"
    elif answer_type == "choice_list":
        format_rule = "[A, B, C]"
    else:
        format_rule = "short text"

    # caption
    lines = []
    for item in items:
        caption_text = caption_map.get(item, "")
        lines.append(f"the caption of {item}: {caption_text}")
    caption = "\n".join(lines)

    prompt = f"""
You are a data science assistant specialized in understanding charts and tables.

You are given the following items: {items}.
Each item has a corresponding caption:
{caption}

Previous Question:
{question_gen}

The chart generated from the previous question is provided as: gen.jpg

Question:
{question_comp}

Instructions:
1. Analyze the provided items and the generated chart (gen.jpg) to answer the Q2 question.
2. Do NOT output reasoning, explanation, or intermediate steps.
3. Output ONLY the final answer.

Output format:
Answer: {format_rule}

Rules:
- Output exactly ONE line.
- The line must start with "Answer:".
- Do not output anything before or after the answer.
"""

    return prompt

In [17]:
# 转换图表格式 压缩图片-> 减少token
def encode_image_to_base64(image_path, max_size=768, quality=70):
    img = Image.open(image_path)

    # 等比例缩小
    img.thumbnail((max_size, max_size))
    
    if img.mode != "RGB":
        img = img.convert("RGB")

    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=quality, optimize=True)

    return base64.b64encode(buffer.getvalue()).decode("utf-8")

In [ ]:
def call_llm_multiround(subject_path, sample):
    
    # 生成提示词
    prompt_q1 = generate_prompt_q1(sample)

    content_list_q1 = []
    path_map = sample["path"]
    

    for key in path_map:
        relative_path = path_map[key]
        full_path = os.path.join(subject_path, relative_path)

        if not os.path.exists(full_path):
            print(f"文件不存在: {full_path}")
            continue

        mime_type = "image/jpeg"
        image_base64 = encode_image_to_base64(full_path)

        content_list_q1.append({
            "type": "image_url",
            "image_url": {
                "url": f"data:{mime_type};base64,{image_base64}"
            }
        })
        
    # 额外加入同目录下的 gen.jpg
    if path_map:
        first_relative_path = next(iter(path_map.values()))
        gen_relative_path = os.path.join(os.path.dirname(first_relative_path), "gen.jpg")
        gen_full_path = os.path.join(subject_path, gen_relative_path)

        if os.path.exists(gen_full_path):
            gen_base64 = encode_image_to_base64(gen_full_path)
            content_list_q1.append({
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{gen_base64}"
                }
            })
        else:
            print(f"gen.jpg 不存在: {gen_full_path}")
        
    content_list_q2 = content_list_q1.copy()

    # 添加文本
    content_list_q1.append({
        "type": "text",
        "text": prompt_q1
    })
    
    # 调用模型
    client = OpenAI(
        base_url="", 
        api_key=""
    )
    
    attempt = 0
    base_sleep = 1
    max_sleep= 60
    MAX_RETRY = 50 # 避免死循环    
    
    while True:
        if attempt > MAX_RETRY:
            raise RuntimeError("LLM retry too many times")

        try:
            # Q1
            # resp_q1 = client.chat.completions.create(
            #     model="Qwen/Qwen3-VL-32B-Instruct",
            #     messages=[{"role": "user", "content": content_list_q1}],
            #     temperature=0
            # )
            # q1_resp = resp_q1.choices[0].message.content
            
            # print(f"q1_resp: {q1_resp}")
            
            
            # Q2
            prompt_q2 = generate_prompt_q2(sample)
            content_list_q2.append({
                "type": "text",
                "text": prompt_q2
            })
            
            resp_q2 = client.chat.completions.create(
                model="",
                messages=[{"role": "user", "content": content_list_q2}],
                temperature=0
            )
            q2_resp = resp_q2.choices[0].message.content
            
            print(f"q2_resp: {q2_resp}")

            return q2_resp
        
        except Exception as e:
            msg = str(e).lower()
            
            if ("401" in msg or "403" in msg or "400" in msg or "invalid" in msg or "not found" in msg):
                raise
            
            sleep_s = min(max_sleep, base_sleep * (2 ** attempt))
            time.sleep(sleep_s)
            
            attempt += 1
    
    

In [ ]:
# 使用deepseek提取答案
def exact_answer_llm(output, answer_type):
    
    # 设计prompt
        # 输出规则 
    if answer_type == "numeric":
        output_format = "answer: {number}. Return exactly one number after 'answer:'. Do not include units, symbols, commas, explanations, or extra text."

    elif answer_type == "numeric_list":
        output_format = "answer: [a, b, c]. Return a list of numbers inside square brackets. No text inside brackets. No explanations."

    elif answer_type == "choice":
        output_format = "answer: {A capital letter}. Return exactly one capital letter after 'answer:'. No punctuation. No brackets. No extra text."

    elif answer_type == "choice_list":
        output_format = "answer: [A, B, C]. Return a list of capital letters inside square brackets. No explanations."
    else:
        output_format = "answer: {text}. Return only the final short text answer after 'answer:'. The answer MUST end with a period '.'. Do not include anything after the period. No explanations or additional commentary."
        
    
    prompt = f"""
You are an answer extraction assistant. Your task is to extract the final answer from a model’s response according to the specified answer type. 
Answer type: {answer_type}
Model response:
{output}
You must extract only the final answer and strictly follow this output format:
{output_format}
Do not include explanations or any additional text.
Return exactly one answer in the required format.
    """
        
    # 调用llm 多次调用 直到成功返回 deepseek
    client = OpenAI(
        base_url="", 
        api_key=""
    )
    
    attempt = 0
    base_sleep=1.0
    max_sleep=60.0
    MAX_RETRY = 50 # 避免死循环
    
    while True:
        if attempt > MAX_RETRY:
            raise RuntimeError("LLM retry too many times")
    
        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": prompt}
            ],
                max_tokens=1024,
                temperature=0,
                stream=False
            )

            output_llm = response.choices[0].message.content
            
            if not output_llm:
                raise ValueError("empty response")
                
            return output_llm
 
        except Exception as e:
            msg = str(e).lower()

            if ("401" in msg) or ("403" in msg) or ("400" in msg) or ("invalid" in msg) or ("not found" in msg):
                raise

            sleep_s = min(max_sleep, base_sleep * (2 ** attempt))
            time.sleep(sleep_s)
            
            attempt += 1

In [21]:
# 测评代码 —— numeric
# 提取数字 -> 计算相对误差 —> 相对误差小于tol则得1分

def evaluate_numeric(output, gt):
    eps_rel = 0.01
    eps_abs = 1e-6
    
    if output is None:
        return False

    output = output.strip()
    
    # 提取答案
    output_llm = exact_answer_llm(output, "numeric")
    
    number_pattern = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"

    # 提取answer:后面的数字
    match = re.search(rf"^answer:\s*({number_pattern})\s*$", output_llm.strip(), re.IGNORECASE)

    if match:
        pred = float(match.group(1))
    else:
        # 情况2：没有answer 则提取第一个数字
        match = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", output_llm)
        
        if not match:
            return False  
        
        pred = float(match.group())

    gt = float(gt)

    tolerance = max(eps_abs, eps_rel * abs(gt))

    return abs(pred - gt) <= tolerance

In [22]:
# 测评代码 —— numeric_list
# 提取数字 -> 计算相对误差 —> 所有相对误差小于tol才算正确

def evaluate_numeric_list(output, gt):
    eps_rel = 0.01
    eps_abs = 1e-6

    if output is None:
        return False
    
    s = output.strip()
    
    output_llm = exact_answer_llm(s, "numeric_list")

    # 取answer后面的list
    number_pattern = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"
    match = re.search(rf"(?is)answer:\s*(\[\s*{number_pattern}(?:\s*,\s*{number_pattern})*\s*\])", output_llm)

    if match:
        list_str = match.group(1)
    else:
        # 提取第一个[...]
        match = re.search(rf"(\[\s*{number_pattern}(?:\s*,\s*{number_pattern})*\s*\])", output_llm)
        if match:
            list_str = match.group(1)
        else:
            return False

    # 解析成list并校验
    try:
        pred = ast.literal_eval(list_str)
    except Exception:
        return False

    if not isinstance(pred, list):
        return False

    if len(pred) != len(gt): # 元素个数不一致则回答错误
        return False

    for pi, gi in zip(pred, gt):
        try:
            pi = float(pi)
            gi = float(gi)
        except Exception:
            return False

        toli = max(eps_abs, eps_rel * abs(gi))
        if abs(pi - gi) > toli:
            return False

    return True
    

In [23]:
# 测评代码 —— text
# 衡量两个字符串的相似程度 用编辑距离衡量

# 规范化字符串
def normalize_text(s):
    if s is None:
        return ""
    
    s = str(s).lower().strip()
    
    # 去掉常见的中英文标点
    punct = string.punctuation + "，。！？；：“”‘’（）【】《》、…—·"
    s = s.translate(str.maketrans("", "", punct))
    
    # 空格合一
    s = re.sub(r"\s+", " ", s).strip()
    return s

# 计算编辑距离
def edit_distance(a, b):
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n
    
    prev = list(range(m + 1))
    for i in range(1, n + 1):
        cur = [i] + [0] * m
        ai = a[i - 1]
        for j in range(1, m + 1):
            cost = 0 if ai == b[j - 1] else 1
            cur[j] = min(
                prev[j] + 1,      # delete
                cur[j - 1] + 1,   # insert
                prev[j - 1] + cost  # substitute
            )
        prev = cur
    return prev[m]

# 测评
def evaluate_text(output, gt):
    threshold = 0.9
    
    if output is None:
        return False
    
    output_llm = exact_answer_llm(output, "text")
    
    # 提取answer: 后面的text 遇到.结束
    m = re.search(r"(?is)^\s*answer:\s*(.+?\.)", output_llm.strip())
    if not m:
        return False

    pred_text = m.group(1).strip()
    
    p = normalize_text(pred_text)
    g = normalize_text(gt)
    
    if len(p) == 0 and len(g) == 0:
        return True
    if len(p) == 0 or len(g) == 0:
        return False
        
    dist = edit_distance(p, g)
    sim = 1.0 - dist / max(len(p), len(g))
    
    return sim >= threshold
    
    

In [24]:
# 测评代码 —— choice
# 提取原则： 如果只有一个大写字母，则该字母为答案，否则提取左右两边都是标点的大写字母 若有多个 则用最后一个

def evaluate_choice(output, gt):
    s = str(output).strip()
    
    output_llm = exact_answer_llm(s, "choice")
    
    # 提取answer: 后面的第一个大小字母
    match = re.search(r"(?is)^\s*answer:\s*([A-Z])\s*(?:\r?\n|$)", output_llm.strip())
    if not match:
        return False

    pred = match.group(1)
    
    return pred == str(gt).strip()


In [25]:
# 测评代码 —— choice_list
def evaluate_choice_list(output, gt):
    if output is None:
        return False
    
    s = str(output).strip()
    
    output_llm = exact_answer_llm(s, "choice")
    
    # 提取 answer: 后面的第一个 [A, B, C]
    match = re.search(r"(?is)^\s*answer:\s*(\[\s*[A-Z](?:\s*,\s*[A-Z])*\s*\])", output_llm.strip())

    if not match:
        return False

    list_str = match.group(1)

    # 解析为Python list
    try:
        pred = ast.literal_eval(list_str)
    except Exception:
        return False

    if not isinstance(pred, list):
        return False

    # 去空格并标准化
    pred = [x.strip() for x in pred]
    
    if not all(re.fullmatch(r"[A-Z]", x) for x in pred):
        return False
    
    return set(pred) == set(gt)
    

In [ ]:
ROOT_PATH = "und_gen"

SUBJECT = "table_to_chart"
JSON_FILE = "gen_to_comp.json"


def evaluate_dataset(subject_path, full_json_path):

    with open(full_json_path, "r", encoding="utf-8") as f:
        samples_data = json.load(f)

    stats = {
        "perceptual": {"score": 0, "count": 0},
        "reasoning": {"score": 0, "count": 0},
        "table to chart": {"score": 0, "count": 0},
        "chart to table": {"score": 0, "count": 0},
    }

    for sample in samples_data:
        print("-----sample-----")
        print(sample)

        output = call_llm_multiround(subject_path, sample)
        gt = sample["answer"]
        answer_type = sample["answer_type"]
        
        if answer_type == "numeric":
            score = evaluate_numeric(output, gt)
        elif answer_type == "numeric_list":
            score = evaluate_numeric_list(output, gt)
        elif answer_type == "text":
            score = evaluate_text(output, gt)
        elif answer_type == "choice":
            score = evaluate_choice(output, gt)
        elif answer_type == "choice_list":
            score = evaluate_choice_list(output, gt)
        question_type = sample["question_type"]

        if question_type in stats:
            stats[question_type]["count"] += 1
            stats[question_type]["score"] += score

    return stats


# ===== 路径 =====
subject_path = os.path.join(ROOT_PATH, SUBJECT)
full_json_path = os.path.join(subject_path, JSON_FILE)

if not os.path.exists(full_json_path):
    raise FileNotFoundError(f"{full_json_path} 不存在")

print(f"\n====== Running {SUBJECT} | {JSON_FILE} ======")

# ===== 运行 =====
stats = evaluate_dataset(subject_path, full_json_path)


# ===== 输出结果 =====
print("\n====== Result ======")

for k in stats:
    total_score = stats[k]["score"]
    total_count = stats[k]["count"]

    avg = total_score / total_count if total_count else 0

    print(f"{k}: Count={total_count}, Avg={avg:.4f}")